<a href="https://colab.research.google.com/github/himanshugajbhiyebhai302-hash/DEEPLEARNING/blob/main/MIT_LAB_2_UNBIASING_PRACTICEpynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import IPython
IPython.display.YouTubeVideo("59bMh59JQDo")

In [ ]:
## Comet ML
!pip install comet_ml --quiet
import comet_ml

from google.colab import userdata
COMET_API_KEY = userdata.get('comet_ml')

!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

In [ ]:
import os
import random
import functools
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from pathlib import Path

#Import torch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

# cuda language
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cudnn.benchmark = True

In [ ]:
CACHE_DIR = Path.home() / ".cache"/"mitdeeplearning"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

#get the training data: both from celebimage and imagenet
path_to_training_data = CACHE_DIR.joinpath("train_face_h5")

if path_to_training_data.exists():
  print(f"Using cache  trainiinng data :{path_to_training_data}")
else:
    print(f"Downloading training data to :{path_to_training_data}")
    url = "https://www.dropbox.com/s/hlz8atheyozp1yx/train_face.h5?dl=1"
    torch.hub.download_url_to_file(url,path_to_training_data)

# Instantiate a TrainingDatasetLoader using the download dataset
channels_last = False
loader = mdl.lab2.TrainingDatasetLoader(
    path_to_training_data, channels_last=channels_last
 )


In [ ]:
number_of_training_examples = loader.get_train_size()
(images, labels) = loader.get_batch(100)

In [ ]:
B, C, H, W = images.shape

In [ ]:
face_images  = images[np.where(labels == 1)[0]].transpose(0,2,3,1)
not_face_images = images[np.where(labels == 0)[0]].transpose(0,2,3,1)

idx_face = 14    # @param {type:"slider", min:0, max:50, step:1}
idx_not_face = 16  # @param {type:"slider", min:0, max:50, step:1}

plt.figure(figsize=(5,5))
plt.subplot(1,2,1)
plt.imshow(face_images[idx_face])
plt.title("Face")
plt.grid(False)

plt.subplot(1,2,2)
plt.title("not face")
plt.imshow(not_face_images[idx_not_face])
plt.grid(False)

plt.show()


sequences = 12
in_channels = images.shape[1]
def make_standard_classification(n_outputs):
  """Create a standard CNN Classifier"""

  # yeh hogyi apni process so apanne,
  # class define ki matlab apna model ke kaam ki process batayi
  class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding=0):
      super().__init__()
      self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
      self.relu = nn.ReLU()
      self.bm = nn.BatchNorm2d(out_channels)

    def forward(self,x):
      x = self.conv(x)
      x = self.relu(x)
      x = self.bm(x)
      return x

   # yeh hogya apna model
  model = nn.Sequential(
      ConvBlock(in_channels,sequences,kernel_size=5,stride=2,padding=2),
      ConvBlock(sequences, 2*sequences,kernel_size=5,stride=2,padding=2),
      ConvBlock(2*sequences, 4*sequences,kernel_size=5,stride=2,padding=2),
      ConvBlock(4*sequences, 6*sequences,kernel_size=5,stride=2,padding=2),
      nn.Flatten(), # Corrected from nn.flatten()
      nn.Linear(H // 16*W // 16*6*sequences, 512),
      nn.ReLU(), # Corrected from nn.ReLU
      nn.Linear(512, n_outputs)

  )

  return model.to(device)

  #Final chapter
standard_classifier = make_standard_classification(n_outputs=1)
print(standard_classifier)


In [ ]:
sequences = 12
in_channels = images.shape[1]

def make_standard_classification(n_outputs):
   """Create a standard CNN Classifier"""

   # yeh hogyi apni process so apanne,
   # class define ki matlab apna model ke kaam ki process batayi
   class ConvBlock(nn.Module):
     def __init__(self, in_channels, out_channels, kernel_size, stride, padding=0):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        self.relu = nn.ReLU()
        self.bm = nn.BatchNorm2d(out_channels)

     def forward(self,x):
       x = self.conv(x)
       x = self.relu(x)
       x = self.bm(x)
       return x

   #yeh hogya apna model
   model = nn.Sequential(
       ConvBlock(in_channels,sequences,kernel_size=5,stride=2,padding=2),
       ConvBlock(sequences, 2 * sequences,kernel_size=5,stride=2,padding=2),
       ConvBlock(2 * sequences, 4 * sequences,kernel_size=5,stride=2,padding=2),
       ConvBlock(4 * sequences, 6 * sequences,kernel_size=5,stride=2,padding=2),
       nn.Flatten(),
       nn.Linear(H // 16 * W // 16 * 6 * sequences, 512),
       nn.ReLU(),
       nn.Linear(512, n_outputs)
   )

   return model.to(device)

#Final chapter
standard_classifier = make_standard_classification(n_outputs=1)
print(standard_classifier)


In [ ]:
def create_experiment(project_name, params):
    # end any prior experiments
    if "experiment" in locals():
        experiment.end()

    # initiate the comet experiment for tracking
    experiment = comet_ml.Experiment(api_key=COMET_API_KEY, project_name=project_name)
    # log our hyperparameters, defined above, to the experiment
    for param, value in params.items():
        experiment.log_parameter(param, value)
    experiment.flush()

    return experiment

In [ ]:
##Train the standarad model
loss_fn = nn.BCEWithLogitsLoss()
##
params = dict(
    batch_size = 32,
    num_epochs = 2,
    learning_rate  = 5e-4 ,
)

experiment = create_experiment("6S191_Lab2_Part2_CNN", params)

optimizer = optim.Adagrad(standard_classifier.parameters(), lr=params["learning_rate"])
# define our optimizer
loss_history = mdl.util.LossHistory(smoothing_factor = 0.99)
plotter  = mdl.util.PeriodicPlotter(sec=2, scale="semilogy")
if hasattr(tqdm, "instances"):
  tqdm.instances.clear()

#set the model to train mode
standard_classifier.train()

def train_step(x,y):
  x =torch.from_numpy(x).to(device)
  y = torch.from_numpy(y).to(device)

  # clear the gradiants
  optimizer.zero_grad()

  #feed the images into the model
  logits = standard_classifier(x) # Call the model here
  # compute the loss
  loss = loss_fn(logits, y)

  #Backpropagation
  loss.backward()
  optimizer.step()

  return loss

# The training loop
step = 0
for epoch in range(params["num_epochs"]):
  for idx in tqdm(range(loader.get_train_size() // params["batch_size"])):
    #Grab a batch of training data and propagation through the network
    x, y = loader.get_batch(params["batch_size"]) # Corrected batch_size
    loss = train_step(x,y) # Corrected function name
    loss_value = loss.detach().cpu().numpy()


    loss_history.append(loss_value)
    plotter.plot(loss_history.get())

    experiment.log_metric("loss", loss_value, step=step)
    step += 1

experiment.end()

In [ ]:
# Set the model for evaluation
standard_classifier.eval()

# TRAINING DATA
# Evaluate on a dataset of CeleBa + Imagenet
(batch_x, batch_y) = loader.get_batch(5000)
batch_x = torch.from_numpy(batch_x).float().to(device)
batch_y = torch.from_numpy(batch_y).float().to(device)

with torch.inference_mode():
    y_pred_logits = standard_classifier(batch_x)
    y_pred_standard = torch.round(torch.sigmoid(y_pred_logits))

    #Accuracy
    acc_standard = torch.mean((batch_y == y_pred_standard).float())

print(
    "Standard CNN accuracy on (potentially biased) training set: {:.4f}".format(
        acc_standard.item()
    )
)



In [ ]:
### Load test dataset and plot examples ###

test_faces = mdl.lab2.get_test_faces(channels_last=channels_last)
keys = ["Light Female", "Light Male", "Dark Female", "Dark Male"]

fig, axs = plt.subplots(1,len(keys),
                        figsize=(7.5, 7.5))
for i, (group, key) in enumerate(zip(test_faces, keys)):
  axs[i].imshow(np.hstack(group).transpose(1,2,0))
  axs[i].set_title(key, fontsize=15)
  axs[i].axis("off")

In [ ]:
### Evaluate the standard CNN on the test data ##
standard_classifier_probs_list = []

with torch.inference_mode():
  for x in test_faces:
    x = torch.from_numpy(np.array(x, dtype=np.float32)).to(device)
    logits = standard_classifier(x)
    probs = torch.sigmoid(logits)
    probs = torch.squeeze(probs, dim=-1)
    standard_classifier_probs_list.append(probs.cpu().numpy())

standard_classifier_probs = np.array(standard_classifier_probs_list)

# Plot the prediction accuracies per demograpic
xx = range(len(keys))
yy = standard_classifier_probs.mean(axis=1)
plt.bar(xx, yy)
plt.xticks(xx,keys)
plt.ylim(max(0, yy.min() - np.ptp(yy) / 2.0), yy.max() + np.ptp(yy) / 2.0)
plt.title("Standard classifier predictions")